
# Balanced two-sample permutation study — multi-distribution H0/H1

Self-contained notebook for:

- Gaussian-equivalence-class approximations (`ordered`, `original`, `balanced`)
- Monte Carlo permutation reference with configurable \(B_{\rm ref}\)
- H0 and H1
- multiple standardized generating distributions
- \(R\) datasets per distribution
- log-space p-value and \(B_{\rm eq}\) computations
- unresolved-reference flag when MC observes \(K=0\)
- progress bars
- checkpointing/resume
- generating-distribution diagnostics
- \(B_{\rm eq}\) and equivalent-time plots, both distribution-specific and combined
- **automatic local/OpenPBS environment detection**
- **cluster-safe persistent output under `/work/<user>/...`**
- **job-local scratch directory under `/scratch_local`**
- **headless Matplotlib output on compute nodes**
- **run metadata, hostname, PBS job ID and total runtime logging**

The theoretical equivalent budget is

$$
B_{\rm eq}
=
\frac{\mathbb E[p(1-p)]}
{\mathbb E[(p^\circ-p)^2]}.
$$

Across \(R\) datasets it is estimated as a **ratio of sums**, never as an average of individual ratios.

## Cluster behavior

When the notebook detects `PBS_JOBID`, it assumes it is running inside an OpenPBS job. It then:

1. uses `PBS_O_WORKDIR` as the submitted project directory;
2. stores persistent checkpoints, CSV summaries, figures and metadata under `/work/<user>/thesis_results/...`;
3. creates a job-specific `/scratch_local/...` directory for optional temporary files;
4. switches Matplotlib to the non-interactive `Agg` backend;
5. saves all figures to disk instead of opening GUI windows.

When run locally, the same notebook stores results relative to the current directory and still saves figures, while also displaying them interactively.


In [ ]:

# ============================================================
# 1. IMPORTS + EXECUTION ENVIRONMENT
import re
# ============================================================

import os
import json
import math
import socket
import shutil
import hashlib
import warnings
from datetime import datetime, timezone
from pathlib import Path
from time import perf_counter

# ------------------------------------------------------------
# Detect OpenPBS before importing pyplot so that cluster jobs
# use a non-interactive backend.
# ------------------------------------------------------------
IS_PBS_JOB = bool(os.environ.get("PBS_JOBID"))
PBS_JOB_ID = os.environ.get("PBS_JOBID", "local")
PBS_QUEUE = os.environ.get("PBS_QUEUE", "local")
USER_NAME = os.environ.get("USER", "unknown")
HOSTNAME = socket.gethostname()

PROJECT_DIR = Path(
    os.environ.get("PBS_O_WORKDIR", Path.cwd())
).resolve()

if IS_PBS_JOB:
    import matplotlib
    matplotlib.use("Agg")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.special import gammaln, logsumexp
from scipy.stats import norm, skew, kurtosis
from tqdm.auto import tqdm

pd.set_option("display.max_columns", 100)
np.set_printoptions(precision=6, suppress=True)

NOTEBOOK_START = perf_counter()
START_UTC = datetime.now(timezone.utc).isoformat()

print("Execution environment")
print("---------------------")
print(f"PBS job detected   = {IS_PBS_JOB}")
print(f"PBS job ID         = {PBS_JOB_ID}")
print(f"Hostname           = {HOSTNAME}")
print(f"Project directory  = {PROJECT_DIR}")


In [ ]:

# ============================================================
# 2. CONFIGURATION
# ============================================================

# Set True for a very quick end-to-end smoke test.
# Set False for the requested benchmark/full pilot.
FAST_SMOKE_TEST = False

if FAST_SMOKE_TEST:
    N_VALUES = [100, 500]
    R = 2
    B_REF = 1_000
    BATCH_SIZE = 250
else:
    # Identical to the local-PC pilot used for the first
    # cluster-vs-PC timing comparison.
    N_VALUES = [100, 500, 1_000, 2_000, 5_000]
    R = 20
    B_REF = 10_000
    BATCH_SIZE = 500

HYPOTHESES = ["H0", "H1"]

# H0: both groups have the same location.
MEAN_X_H0 = 0.0
MEAN_Y_H0 = 0.0

# H1: same shape/scale family, but a location difference.
MEAN_X_H1 = 0.0
MEAN_Y_H1 = 0.5

STANDARD_DEVIATION = 1.0
TAIL = "two-sided"

# Descriptive only. A reference is strictly unresolved only at K=0.
MIN_EXTREME_COUNT = 100

MASTER_SEED = 20260907

print("\nConfiguration")
print("-------------")
print(f"N_VALUES          = {N_VALUES}")
print(f"R                 = {R}")
print(f"B_REF             = {B_REF:,}")
print(f"BATCH_SIZE        = {BATCH_SIZE}")
print(f"HYPOTHESES        = {HYPOTHESES}")
print(f"H1 shift          = {MEAN_Y_H1 - MEAN_X_H1}")


In [ ]:

# ============================================================
# 3. GENERATING DISTRIBUTIONS
# ============================================================
#
# Every base family is standardized to mean 0 and variance 1
# before the H0/H1 location shift is added.
# ============================================================

DISTRIBUTIONS = [
    {
        "label": "Gaussian",
        "family": "normal",
        "params": {},
    },
    {
        "label": "Exponential",
        "family": "exponential",
        "params": {},
    },
    {
        "label": "Gaussian mixture",
        "family": "gaussian_mixture",
        "params": {
            "component_mean": 1.5,
            "component_sd": 0.5,
        },
    },
    {
        "label": "Gamma (shape=2)",
        "family": "gamma",
        "params": {
            "shape": 2.0,
        },
    },
    {
        "label": "Student t (df=5)",
        "family": "student_t",
        "params": {
            "df": 5.0,
        },
    },
    {
        "label": "Laplace",
        "family": "laplace",
        "params": {},
    },
]


def draw_standardized_sample(rng, size, family, params=None):
    params = {} if params is None else dict(params)
    family = family.lower()

    if family in {"normal", "gaussian"}:
        z = rng.normal(0.0, 1.0, size=size)

    elif family == "exponential":
        # Exp(1): mean=1, variance=1.
        z = rng.exponential(scale=1.0, size=size) - 1.0

    elif family == "gamma":
        shape = float(params.get("shape", 2.0))
        if shape <= 0:
            raise ValueError("Gamma shape must be positive.")
        raw = rng.gamma(shape=shape, scale=1.0, size=size)
        z = (raw - shape) / np.sqrt(shape)

    elif family in {"student_t", "student-t", "t"}:
        df = float(params.get("df", 5.0))
        if df <= 2:
            raise ValueError("Student-t df must exceed 2 for finite variance.")
        raw = rng.standard_t(df, size=size)
        z = raw / np.sqrt(df / (df - 2.0))

    elif family == "laplace":
        # Var Laplace(0,b)=2b^2 => b=1/sqrt(2) gives unit variance.
        z = rng.laplace(0.0, 1.0 / np.sqrt(2.0), size=size)

    elif family in {"gaussian_mixture", "normal_mixture", "mixture"}:
        mu = float(params.get("component_mean", 1.5))
        sigma = float(params.get("component_sd", 0.5))
        if sigma <= 0:
            raise ValueError("Mixture component_sd must be positive.")

        component = rng.integers(0, 2, size=size)
        locations = np.where(component == 0, -mu, mu)
        raw = rng.normal(loc=locations, scale=sigma, size=size)

        # Equal symmetric mixture: mean=0, var=mu^2+sigma^2.
        z = raw / np.sqrt(mu**2 + sigma**2)

    else:
        raise ValueError(f"Unknown distribution family: {family}")

    return np.asarray(z, dtype=float)


def generate_nested_dataset_path(
    n_values,
    hypothesis,
    data_seed,
    family,
    params=None,
    mean_x_h0=0.0,
    mean_y_h0=0.0,
    mean_x_h1=0.0,
    mean_y_h1=0.5,
    standard_deviation=1.0,
):
    """
    Generate one nested path for a single replicate:
    max(n_values) observations per group are drawn once, then
    prefixes are reused for smaller n.
    """
    if hypothesis not in {"H0", "H1"}:
        raise ValueError("hypothesis must be 'H0' or 'H1'.")

    rng = np.random.default_rng(data_seed)
    n_max = int(max(n_values))

    base_x = draw_standardized_sample(rng, n_max, family, params)
    base_y = draw_standardized_sample(rng, n_max, family, params)

    if hypothesis == "H0":
        mean_x, mean_y = mean_x_h0, mean_y_h0
    else:
        mean_x, mean_y = mean_x_h1, mean_y_h1

    x_full = mean_x + standard_deviation * base_x
    y_full = mean_y + standard_deviation * base_y

    return {
        int(n): (x_full[:int(n)].copy(), y_full[:int(n)].copy())
        for n in n_values
    }


# ============================================================
# RUN ID + OUTPUT PATHS
# ============================================================
# The run signature depends on the scientifically relevant
# configuration. Changing B_REF, R, N_VALUES, hypotheses,
# distribution definitions, effect size, etc. automatically creates
# a different folder, preventing accidental stale-checkpoint reuse.

RUN_CONFIG = {
    "n_values": [int(x) for x in N_VALUES],
    "R": int(R),
    "B_ref": int(B_REF),
    "batch_size": int(BATCH_SIZE),
    "hypotheses": list(HYPOTHESES),
    "mean_x_h0": float(MEAN_X_H0),
    "mean_y_h0": float(MEAN_Y_H0),
    "mean_x_h1": float(MEAN_X_H1),
    "mean_y_h1": float(MEAN_Y_H1),
    "standard_deviation": float(STANDARD_DEVIATION),
    "tail": TAIL,
    "min_extreme_count": int(MIN_EXTREME_COUNT),
    "master_seed": int(MASTER_SEED),
    "distributions": DISTRIBUTIONS,
}

RUN_HASH = hashlib.sha256(
    json.dumps(RUN_CONFIG, sort_keys=True).encode("utf-8")
).hexdigest()[:10]

MODE_TAG = "smoke" if FAST_SMOKE_TEST else "full"
RUN_NAME = f"multidist_{MODE_TAG}_B{B_REF}_R{R}_{RUN_HASH}"

# Scrive nella DIRECTORY CORRENTE: submit_chain.sh (sul cluster) o le
# istruzioni del README (in locale) creano gia' una cartella contenitore
# univoca (nome_timestamp) e ci fanno cd DENTRO prima di eseguire questo
# script. Il notebook non deve occuparsi ne' di nome ne' di timestamp
# della propria cartella di output, solo scrivere relativo alla cwd.
OUTPUT_DIR = Path(".")
FIGURE_DIR = OUTPUT_DIR / "figures"
TEMP_DIR = OUTPUT_DIR / "scratch"

FIGURE_DIR.mkdir(parents=True, exist_ok=True)
TEMP_DIR.mkdir(parents=True, exist_ok=True)

CHECKPOINT_RAW = OUTPUT_DIR / "raw_results_checkpoint.csv"
SUMMARY_FILE = OUTPUT_DIR / "scenario_summary.csv"
GENERATION_FILE = OUTPUT_DIR / "generation_manifest.csv"
METADATA_FILE = OUTPUT_DIR / "run_metadata.json"

print(f"Run name          = {RUN_NAME}")
print(f"Persistent output = {OUTPUT_DIR}")
print(f"Figure directory  = {FIGURE_DIR}")
print(f"Temporary scratch = {TEMP_DIR}")
print(f"Checkpoint        = {CHECKPOINT_RAW}")


def _safe_filename(text):
    """Filesystem-safe compact filename component."""
    text = str(text).strip().lower()
    text = re.sub(r"[^a-z0-9]+", "_", text)
    return text.strip("_") or "plot"


def save_or_show_figure(fig, filename, dpi=180):
    """
    Always save the figure. Show it only outside PBS.
    """
    path = FIGURE_DIR / filename
    fig.savefig(path, dpi=dpi, bbox_inches="tight")

    if IS_PBS_JOB:
        plt.close(fig)
    else:
        plt.show()

    return path


In [ ]:

# ============================================================
# 4. CORE STATISTICS + MC REFERENCE
# ============================================================

def difference_in_means(x, y):
    return float(np.mean(x) - np.mean(y))


def is_extreme(permuted_statistics, observed_statistic, tail="two-sided"):
    if tail == "two-sided":
        return np.abs(permuted_statistics) >= abs(observed_statistic)
    if tail == "greater":
        return permuted_statistics >= observed_statistic
    if tail == "less":
        return permuted_statistics <= observed_statistic
    raise ValueError("tail must be 'two-sided', 'greater', or 'less'.")


def classify_mc_resolution(K, min_extreme_count=100):
    if K == 0:
        return "unresolved"
    if K < min_extreme_count:
        return "low_resolution"
    return "well_resolved"


def monte_carlo_permutation_test(
    x,
    y,
    B,
    rng,
    batch_size=500,
    tail="two-sided",
    min_extreme_count=100,
):
    """
    Balanced two-sample permutation MC.

    Each row chooses a uniform size-n subset of the 2n pooled
    observations via random keys + argpartition.
    """
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    if len(x) != len(y):
        raise ValueError("This balanced-study MC requires len(x) == len(y).")

    n = len(x)
    pooled = np.concatenate([x, y])
    total_sum = float(pooled.sum())
    observed = difference_in_means(x, y)

    extreme_count = 0
    completed = 0
    elapsed = 0.0

    while completed < B:
        current_batch = min(batch_size, B - completed)

        start = perf_counter()

        keys = rng.random((current_batch, 2 * n))
        first_idx = np.argpartition(
            keys, kth=n - 1, axis=1
        )[:, :n]

        first_sum = pooled[first_idx].sum(axis=1)

        # For equal groups:
        # mean(group1)-mean(group2) = (2*sum(group1)-sum(pool))/n
        perm_stats = (2.0 * first_sum - total_sum) / n

        extreme_count += int(
            is_extreme(perm_stats, observed, tail).sum()
        )

        elapsed += perf_counter() - start
        completed += current_batch

    K = int(extreme_count)

    if K > 0:
        log_p = float(np.log(K) - np.log(B))
        p_value = float(K / B)
        surprisal = float(-log_p / np.log(10.0))
    else:
        log_p = float("-inf")
        p_value = 0.0
        surprisal = float("nan")

    status = classify_mc_resolution(K, min_extreme_count)

    # Exact 95% upper bound for p when K=0:
    zero_count_upper_95 = (
        float(1.0 - 0.05 ** (1.0 / B))
        if K == 0 else float("nan")
    )
    zero_count_surprisal_lower_95 = (
        float(-np.log10(zero_count_upper_95))
        if K == 0 else float("nan")
    )

    return {
        "p_value": p_value,
        "log_p_value": log_p,
        "surprisal": surprisal,
        "extreme_count": K,
        "B": int(B),
        "observed_statistic": observed,
        "elapsed_seconds": float(elapsed),
        "time_per_permutation": float(elapsed / B),
        "resolution_status": status,
        "one_hit_p_resolution": float(1.0 / B),
        "one_hit_surprisal_limit": float(np.log10(B)),
        "zero_count_upper_95": zero_count_upper_95,
        "zero_count_surprisal_lower_95": zero_count_surprisal_lower_95,
    }


In [ ]:

# ============================================================
# 5. EQUIVALENCE-CLASS GAUSSIAN APPROXIMATIONS
# ============================================================

def ec_log_weights(n):
    k = np.arange(n + 1, dtype=int)

    log_choose_n_k = (
        gammaln(n + 1)
        - gammaln(k + 1)
        - gammaln(n - k + 1)
    )

    log_weights = (
        2.0 * log_choose_n_k
        - (
            gammaln(2 * n + 1)
            - 2.0 * gammaln(n + 1)
        )
    )

    # Numerical normalization.
    log_weights -= logsumexp(log_weights)

    return k, log_weights


def gaussian_class_log_tail_probability(
    observed_statistic,
    class_mean,
    class_variance,
    tail="two-sided",
):
    """
    Gaussian exceedance probability computed directly in log space.
    """
    observed_statistic = float(observed_statistic)
    class_mean = float(class_mean)
    class_variance = float(max(class_variance, 0.0))

    if class_variance == 0.0:
        if tail == "two-sided":
            event = abs(class_mean) >= abs(observed_statistic)
        elif tail == "greater":
            event = class_mean >= observed_statistic
        elif tail == "less":
            event = class_mean <= observed_statistic
        else:
            raise ValueError("Invalid tail.")
        return 0.0 if event else float("-inf")

    sd = math.sqrt(class_variance)

    if tail == "two-sided":
        threshold = abs(observed_statistic)

        z_lower = (-threshold - class_mean) / sd
        z_upper = ( threshold - class_mean) / sd

        log_lower = norm.logcdf(z_lower)
        log_upper = norm.logsf(z_upper)

        log_prob = float(np.logaddexp(log_lower, log_upper))

    elif tail == "greater":
        z = (observed_statistic - class_mean) / sd
        log_prob = float(norm.logsf(z))

    elif tail == "less":
        z = (observed_statistic - class_mean) / sd
        log_prob = float(norm.logcdf(z))

    else:
        raise ValueError("Invalid tail.")

    return min(log_prob, 0.0)


def ec_gaussian_approximation_from_split(
    original_x,
    original_y,
    reference_a,
    reference_b,
    method_name,
    tail="two-sided",
):
    original_x = np.asarray(original_x, dtype=float)
    original_y = np.asarray(original_y, dtype=float)
    reference_a = np.asarray(reference_a, dtype=float)
    reference_b = np.asarray(reference_b, dtype=float)

    n = len(original_x)

    if len(original_y) != n:
        raise ValueError("Balanced EC approximation requires equal group sizes.")
    if len(reference_a) != n or len(reference_b) != n:
        raise ValueError("Reference split must contain n observations per group.")

    # Sanity check: reference split must be exactly the same pooled data.
    original_pool = np.sort(np.concatenate([original_x, original_y]))
    reference_pool = np.sort(np.concatenate([reference_a, reference_b]))

    if not np.allclose(original_pool, reference_pool, rtol=0.0, atol=1e-12):
        raise ValueError("Reference split must contain exactly the pooled observations.")

    observed = difference_in_means(original_x, original_y)

    k, log_weights = ec_log_weights(n)

    mean_a = float(np.mean(reference_a))
    mean_b = float(np.mean(reference_b))

    var_a = float(np.var(reference_a, ddof=1))
    var_b = float(np.var(reference_b, ddof=1))

    class_means = (
        (2.0 * k - n) / n
    ) * (mean_a - mean_b)

    class_variances = (
        4.0 * k * (n - k) / (n**3)
    ) * (var_a + var_b)

    class_log_tails = np.array(
        [
            gaussian_class_log_tail_probability(
                observed,
                mu,
                variance,
                tail=tail,
            )
            for mu, variance in zip(class_means, class_variances)
        ],
        dtype=float,
    )

    log_p = float(
        logsumexp(log_weights + class_log_tails)
    )
    log_p = min(log_p, 0.0)

    # Only a display representation; log_p remains authoritative.
    p_value = float(np.exp(log_p))
    surprisal = float(-log_p / np.log(10.0))

    return {
        "method": method_name,
        "p_value": p_value,
        "log_p_value": log_p,
        "surprisal": surprisal,
        "observed_statistic": observed,
    }


def build_original_split(x, y):
    return np.asarray(x, float).copy(), np.asarray(y, float).copy()


def build_ordered_split(x, y):
    n = len(x)
    pooled = np.sort(np.concatenate([x, y]))
    return pooled[-n:].copy(), pooled[:n].copy()


def build_balanced_split(x, y):
    """
    Pair the lower half with the reversed upper half and assign
    whole pairs alternately to the two artificial reference groups.

    Requires even n so each artificial group receives n observations.
    """
    n = len(x)

    if n % 2 != 0:
        raise ValueError("Balanced reference split requires even n.")

    pooled = np.sort(np.concatenate([x, y]))
    low = pooled[:n]
    high = pooled[n:][::-1]

    a_parts = []
    b_parts = []

    for i in range(n):
        pair = np.array([low[i], high[i]], dtype=float)
        if i % 2 == 0:
            a_parts.append(pair)
        else:
            b_parts.append(pair)

    a = np.concatenate(a_parts)
    b = np.concatenate(b_parts)

    if len(a) != n or len(b) != n:
        raise RuntimeError("Balanced split construction produced wrong sizes.")

    return a, b


def ordered_ec_result(x, y, tail="two-sided"):
    a, b = build_ordered_split(x, y)
    return ec_gaussian_approximation_from_split(
        x, y, a, b, "ordered", tail=tail
    )


def original_ec_result(x, y, tail="two-sided"):
    a, b = build_original_split(x, y)
    return ec_gaussian_approximation_from_split(
        x, y, a, b, "original", tail=tail
    )


def balanced_ec_result(x, y, tail="two-sided"):
    a, b = build_balanced_split(x, y)
    return ec_gaussian_approximation_from_split(
        x, y, a, b, "balanced", tail=tail
    )


EC_ESTIMATORS = {
    "ordered": ordered_ec_result,
    "original": original_ec_result,
    "balanced": balanced_ec_result,
}


In [ ]:

# ============================================================
# 6. LOG-SPACE B_eq HELPERS
# ============================================================

def log_abs_exp_difference(log_a, log_b):
    """
    Stable log |exp(log_a) - exp(log_b)|.
    """
    log_a = float(log_a)
    log_b = float(log_b)

    if np.isneginf(log_a) and np.isneginf(log_b):
        return float("-inf")

    if log_a == log_b:
        return float("-inf")

    hi = max(log_a, log_b)
    lo = min(log_a, log_b)

    if np.isneginf(lo):
        return hi

    return float(
        hi + np.log(-np.expm1(lo - hi))
    )


def log_p_one_minus_p(log_p):
    """
    Stable log[p(1-p)] from log(p).
    """
    log_p = float(log_p)

    if np.isneginf(log_p):
        return float("-inf")

    if log_p >= 0.0:
        # p=1 gives p(1-p)=0.
        return float("-inf")

    # Stable log(1-exp(log_p)).
    return float(log_p + np.log(-np.expm1(log_p)))


def safe_exp(log_x):
    """
    Convert back to ordinary scale only for display/plotting.
    """
    if np.isnan(log_x):
        return float("nan")
    if np.isposinf(log_x):
        return float("inf")
    if np.isneginf(log_x):
        return 0.0

    max_log = np.log(np.finfo(float).max)
    return float(np.exp(log_x)) if log_x < max_log else float("inf")


In [ ]:

# ============================================================
# 7. EXPERIMENT RUNNER WITH CHECKPOINTING + PROGRESS BAR
# ============================================================

def deterministic_seed(*parts):
    """
    Stable 32-bit seed from integer components.
    """
    ss = np.random.SeedSequence([int(p) for p in parts])
    return int(ss.generate_state(1, dtype=np.uint32)[0])


def _distribution_param_string(params):
    return json.dumps(params, sort_keys=True)


def _completed_scenario_keys(raw):
    """
    A scenario is complete only when all three methods are present.
    Key: (hypothesis, distribution, run, n, B_ref)
    """
    if raw.empty:
        return set()

    counts = (
        raw.groupby(
            ["hypothesis", "distribution", "run", "n", "B_ref"]
        )["method"]
        .nunique()
    )

    return {
        tuple(idx)
        for idx, count in counts.items()
        if count == len(EC_ESTIMATORS)
    }


def run_multidistribution_study(
    distributions,
    hypotheses,
    n_values,
    R,
    B_ref,
    batch_size,
    master_seed,
    tail="two-sided",
    min_extreme_count=100,
    mean_x_h0=0.0,
    mean_y_h0=0.0,
    mean_x_h1=0.0,
    mean_y_h1=0.5,
    standard_deviation=1.0,
    checkpoint_path=CHECKPOINT_RAW,
    resume=True,
):
    """
    Run H0/H1 × distribution × R × n.

    - one nested dataset path per (H, distribution, run)
    - one MC reference per (H, distribution, run, n), shared by all formulas
    - formula and B_eq ingredients stored in log space
    - K=0 => unresolved reference; no B_eq point is manufactured
    - checkpoint written after every replicate path
    """

    checkpoint_path = Path(checkpoint_path)

    if resume and checkpoint_path.exists():
        raw = pd.read_csv(checkpoint_path)

        # Refuse to silently resume from an incompatible checkpoint.
        required = {"B_ref", "method", "hypothesis", "distribution", "run", "n"}
        missing = required.difference(raw.columns)
        if missing:
            raise RuntimeError(
                f"Checkpoint is missing required columns: {sorted(missing)}"
            )

        existing_b = set(raw["B_ref"].dropna().astype(int).unique())
        if existing_b and existing_b != {int(B_ref)}:
            raise RuntimeError(
                f"Checkpoint B_ref={sorted(existing_b)} does not match current "
                f"B_ref={B_ref}. Use the configuration-specific output folder."
            )

        print(
            f"Resuming from checkpoint with {len(raw):,} existing rows."
        )
    else:
        raw = pd.DataFrame()

    completed = _completed_scenario_keys(raw)

    total_scenarios = (
        len(hypotheses)
        * len(distributions)
        * R
        * len(n_values)
    )

    pbar = tqdm(
        total=total_scenarios,
        desc="Distribution study",
        unit="scenario",
        dynamic_ncols=True,
    )

    # Advance bar over already complete work.
    pbar.update(len(completed))

    new_rows = []

    try:
        for h_idx, hypothesis in enumerate(hypotheses):

            for d_idx, spec in enumerate(distributions):

                label = spec["label"]
                family = spec["family"]
                params = spec.get("params", {})

                for run in range(R):

                    data_seed = deterministic_seed(
                        master_seed, 1000, h_idx, d_idx, run
                    )

                    datasets = generate_nested_dataset_path(
                        n_values=n_values,
                        hypothesis=hypothesis,
                        data_seed=data_seed,
                        family=family,
                        params=params,
                        mean_x_h0=mean_x_h0,
                        mean_y_h0=mean_y_h0,
                        mean_x_h1=mean_x_h1,
                        mean_y_h1=mean_y_h1,
                        standard_deviation=standard_deviation,
                    )

                    replicate_added = False

                    for n_idx, n in enumerate(n_values):

                        key = (
                            hypothesis,
                            label,
                            run,
                            int(n),
                            int(B_ref),
                        )

                        if key in completed:
                            continue

                        x, y = datasets[int(n)]

                        mc_seed = deterministic_seed(
                            master_seed,
                            2000,
                            h_idx,
                            d_idx,
                            run,
                            n_idx,
                        )
                        mc_rng = np.random.default_rng(mc_seed)

                        mc = monte_carlo_permutation_test(
                            x=x,
                            y=y,
                            B=B_ref,
                            rng=mc_rng,
                            batch_size=batch_size,
                            tail=tail,
                            min_extreme_count=min_extreme_count,
                        )

                        for method, estimator in EC_ESTIMATORS.items():

                            start = perf_counter()
                            formula = estimator(x, y, tail=tail)
                            formula_elapsed = perf_counter() - start

                            if mc["extreme_count"] == 0:
                                # Strict rule agreed for the project.
                                log_mc_numerator = float("-inf")
                                log_formula_sq_error = float("nan")
                                log_B_eq_individual = float("nan")
                                beq_status = "unresolved_reference"

                            else:
                                log_mc_numerator = log_p_one_minus_p(
                                    mc["log_p_value"]
                                )

                                log_abs_error = log_abs_exp_difference(
                                    formula["log_p_value"],
                                    mc["log_p_value"],
                                )

                                log_formula_sq_error = (
                                    2.0 * log_abs_error
                                )

                                if np.isneginf(log_formula_sq_error):
                                    log_B_eq_individual = float("inf")
                                else:
                                    log_B_eq_individual = (
                                        log_mc_numerator
                                        - log_formula_sq_error
                                    )

                                beq_status = "resolved"

                            new_rows.append(
                                {
                                    "hypothesis": hypothesis,
                                    "distribution": label,
                                    "distribution_family": family,
                                    "distribution_params": _distribution_param_string(params),

                                    "run": int(run),
                                    "n": int(n),

                                    "method": method,

                                    "data_seed": int(data_seed),
                                    "mc_seed": int(mc_seed),

                                    "B_ref": int(B_ref),

                                    "observed_statistic": mc["observed_statistic"],

                                    "p_ref": mc["p_value"],
                                    "log_p_ref": mc["log_p_value"],
                                    "surprisal_ref": mc["surprisal"],

                                    "mc_extreme_count": mc["extreme_count"],
                                    "mc_resolution_status": mc["resolution_status"],

                                    "mc_one_hit_p_resolution": mc["one_hit_p_resolution"],
                                    "mc_one_hit_surprisal_limit": mc["one_hit_surprisal_limit"],

                                    "mc_zero_count_upper_95": mc["zero_count_upper_95"],
                                    "mc_zero_count_surprisal_lower_95": mc["zero_count_surprisal_lower_95"],

                                    "mc_elapsed_seconds": mc["elapsed_seconds"],
                                    "mc_time_per_permutation": mc["time_per_permutation"],

                                    "p_formula": formula["p_value"],
                                    "log_p_formula": formula["log_p_value"],
                                    "surprisal_formula": formula["surprisal"],
                                    "formula_time_seconds": float(formula_elapsed),

                                    "log_mc_numerator": log_mc_numerator,
                                    "log_formula_squared_error": log_formula_sq_error,
                                    "log_B_eq_individual": log_B_eq_individual,
                                    "B_eq_status": beq_status,
                                }
                            )

                        completed.add(key)
                        replicate_added = True
                        pbar.update(1)

                    # Checkpoint after each replicate path.
                    if replicate_added and new_rows:
                        addition = pd.DataFrame(new_rows)
                        raw = pd.concat([raw, addition], ignore_index=True)
                        raw.to_csv(checkpoint_path, index=False)
                        new_rows = []

    finally:
        pbar.close()

    if new_rows:
        raw = pd.concat([raw, pd.DataFrame(new_rows)], ignore_index=True)
        raw.to_csv(checkpoint_path, index=False)

    raw = raw.sort_values(
        ["hypothesis", "distribution", "run", "n", "method"]
    ).reset_index(drop=True)

    return raw


In [ ]:

# ============================================================
# 8. AGGREGATION ACROSS R DATASETS
# ============================================================

def summarize_distribution_study(raw, strict_unresolved=True):
    """
    For each (H, distribution, n, method):

        B_eq =
            sum_r p_r(1-p_r)
            --------------------------
            sum_r (p^o_r - p_r)^2

    The two sums are evaluated in log space.

    strict_unresolved=True:
        if even one of the R references has K=0, the scenario
        receives no point estimate of B_eq.
    """

    rows = []

    group_cols = [
        "hypothesis",
        "distribution",
        "distribution_family",
        "distribution_params",
        "n",
        "method",
    ]

    for key, group in raw.groupby(group_cols, sort=True):

        (
            hypothesis,
            distribution,
            family,
            params,
            n,
            method,
        ) = key

        # MC fields repeat across methods, but within a method group
        # there is exactly one row per replicate.
        unresolved_count = int(
            (group["mc_extreme_count"] == 0).sum()
        )

        low_resolution_count = int(
            (group["mc_resolution_status"] == "low_resolution").sum()
        )

        well_resolved_count = int(
            (group["mc_resolution_status"] == "well_resolved").sum()
        )

        R_here = len(group)

        can_estimate = (
            unresolved_count == 0
            if strict_unresolved
            else (R_here - unresolved_count) > 0
        )

        if can_estimate:
            usable = group.loc[
                group["mc_extreme_count"] > 0
            ].copy()

            log_num_sum = float(
                logsumexp(
                    usable["log_mc_numerator"].to_numpy(float)
                )
            )

            log_den_sum = float(
                logsumexp(
                    usable["log_formula_squared_error"].to_numpy(float)
                )
            )

            if np.isneginf(log_den_sum):
                log_B_eq = float("inf")
            else:
                log_B_eq = float(log_num_sum - log_den_sum)

            aggregated_B_eq = safe_exp(log_B_eq)

            mc_cost = float(
                np.median(
                    usable["mc_time_per_permutation"].to_numpy(float)
                )
            )

            if mc_cost > 0 and np.isfinite(log_B_eq):
                log_equiv_time = float(
                    log_B_eq + np.log(mc_cost)
                )
                equiv_time = safe_exp(log_equiv_time)

            elif np.isposinf(log_B_eq):
                log_equiv_time = float("inf")
                equiv_time = float("inf")

            else:
                log_equiv_time = float("nan")
                equiv_time = float("nan")

            status = (
                "resolved"
                if unresolved_count == 0
                else "partial_reference_only"
            )

        else:
            log_num_sum = float("nan")
            log_den_sum = float("nan")
            log_B_eq = float("nan")
            aggregated_B_eq = float("nan")
            mc_cost = float("nan")
            log_equiv_time = float("nan")
            equiv_time = float("nan")
            status = "unresolved_reference"

        rows.append(
            {
                "hypothesis": hypothesis,
                "distribution": distribution,
                "distribution_family": family,
                "distribution_params": params,
                "n": int(n),
                "method": method,

                "R": int(R_here),

                "well_resolved_references": well_resolved_count,
                "low_resolution_references": low_resolution_count,
                "unresolved_references": unresolved_count,

                "B_eq_status": status,

                "log_numerator_sum": log_num_sum,
                "log_denominator_sum": log_den_sum,

                "log_aggregated_B_eq": log_B_eq,
                "aggregated_B_eq": aggregated_B_eq,

                "median_mc_time_per_permutation": mc_cost,

                "log_equivalent_mc_time_seconds": log_equiv_time,
                "equivalent_mc_time_seconds": equiv_time,

                "median_formula_time_seconds": float(
                    np.median(
                        group["formula_time_seconds"].to_numpy(float)
                    )
                ),
            }
        )

    summary = pd.DataFrame(rows)

    return summary.sort_values(
        ["hypothesis", "distribution", "method", "n"]
    ).reset_index(drop=True)


In [ ]:

# ============================================================
# 9. GENERATING-DATASET MANIFEST + DIAGNOSTIC PLOTS
# ============================================================

def build_generation_manifest(
    raw,
    distributions,
    n_values,
    mean_x_h0=0.0,
    mean_y_h0=0.0,
    mean_x_h1=0.0,
    mean_y_h1=0.5,
    standard_deviation=1.0,
):
    """
    One row per (hypothesis, distribution, run), using the
    max-n generated sample to summarize what was actually drawn.
    """

    spec_lookup = {
        spec["label"]: spec
        for spec in distributions
    }

    base = (
        raw[
            [
                "hypothesis",
                "distribution",
                "run",
                "data_seed",
            ]
        ]
        .drop_duplicates()
        .sort_values(
            ["hypothesis", "distribution", "run"]
        )
    )

    n_max = int(max(n_values))
    rows = []

    for _, row in base.iterrows():
        hypothesis = row["hypothesis"]
        label = row["distribution"]
        run = int(row["run"])
        data_seed = int(row["data_seed"])

        spec = spec_lookup[label]

        datasets = generate_nested_dataset_path(
            n_values=[n_max],
            hypothesis=hypothesis,
            data_seed=data_seed,
            family=spec["family"],
            params=spec.get("params", {}),
            mean_x_h0=mean_x_h0,
            mean_y_h0=mean_y_h0,
            mean_x_h1=mean_x_h1,
            mean_y_h1=mean_y_h1,
            standard_deviation=standard_deviation,
        )

        x, y = datasets[n_max]

        rows.append(
            {
                "hypothesis": hypothesis,
                "distribution": label,
                "run": run,
                "data_seed": data_seed,
                "n_max": n_max,

                "x_mean": float(np.mean(x)),
                "y_mean": float(np.mean(y)),

                "x_sd": float(np.std(x, ddof=1)),
                "y_sd": float(np.std(y, ddof=1)),

                "x_skewness": float(skew(x, bias=False)),
                "y_skewness": float(skew(y, bias=False)),

                "x_excess_kurtosis": float(kurtosis(x, fisher=True, bias=False)),
                "y_excess_kurtosis": float(kurtosis(y, fisher=True, bias=False)),
            }
        )

    return pd.DataFrame(rows)


def plot_generated_distributions(
    manifest,
    distributions,
    hypothesis,
    n_values,
    mean_x_h0=0.0,
    mean_y_h0=0.0,
    mean_x_h1=0.0,
    mean_y_h1=0.5,
    standard_deviation=1.0,
    bins=70,
):
    """
    One panel per generating family.
    Every thin line is one of the R realized datasets at max(n).
    X = solid thin lines, Y = dashed thin lines.
    Thick lines = average empirical densities.
    """

    subset = manifest.loc[
        manifest["hypothesis"] == hypothesis
    ].copy()

    n_max = int(max(n_values))

    n_dist = len(distributions)
    n_cols = 2
    n_rows = int(np.ceil(n_dist / n_cols))

    fig, axes = plt.subplots(
        n_rows,
        n_cols,
        figsize=(14, 4.2 * n_rows),
    )

    axes = np.atleast_1d(axes).ravel()

    spec_lookup = {
        spec["label"]: spec
        for spec in distributions
    }

    for ax, spec in zip(axes, distributions):
        label = spec["label"]

        rows = subset.loc[
            subset["distribution"] == label
        ].sort_values("run")

        regenerated = []
        pooled_for_range = []

        for _, row in rows.iterrows():
            datasets = generate_nested_dataset_path(
                n_values=[n_max],
                hypothesis=hypothesis,
                data_seed=int(row["data_seed"]),
                family=spec["family"],
                params=spec.get("params", {}),
                mean_x_h0=mean_x_h0,
                mean_y_h0=mean_y_h0,
                mean_x_h1=mean_x_h1,
                mean_y_h1=mean_y_h1,
                standard_deviation=standard_deviation,
            )

            x, y = datasets[n_max]
            regenerated.append((x, y))
            pooled_for_range.extend([x, y])

        all_values = np.concatenate(pooled_for_range)

        lo = float(np.quantile(all_values, 0.002))
        hi = float(np.quantile(all_values, 0.998))

        edges = np.linspace(lo, hi, bins + 1)
        centers = 0.5 * (edges[:-1] + edges[1:])

        x_densities = []
        y_densities = []

        for x, y in regenerated:
            hx, _ = np.histogram(x, bins=edges, density=True)
            hy, _ = np.histogram(y, bins=edges, density=True)

            x_densities.append(hx)
            y_densities.append(hy)

            ax.plot(
                centers, hx,
                linewidth=0.7,
                alpha=0.12,
            )
            ax.plot(
                centers, hy,
                linewidth=0.7,
                alpha=0.12,
                linestyle="--",
            )

        ax.plot(
            centers,
            np.mean(x_densities, axis=0),
            linewidth=2.4,
            label="X mean empirical density",
        )
        ax.plot(
            centers,
            np.mean(y_densities, axis=0),
            linewidth=2.4,
            linestyle="--",
            label="Y mean empirical density",
        )

        ax.set_title(label)
        ax.set_xlabel("Generated value")
        ax.set_ylabel("Density")
        ax.grid(True, alpha=0.2)
        ax.legend(fontsize=8)

    for ax in axes[n_dist:]:
        ax.set_visible(False)

    fig.suptitle(
        f"Generated datasets under {hypothesis} — all R realizations at n={n_max}",
        fontsize=14,
        y=1.01,
    )

    plt.tight_layout()
    save_or_show_figure(
        fig,
        f"generated_distributions_{_safe_filename(hypothesis)}.png",
    )


In [ ]:

# ============================================================
# 10. RESULT PLOTS
# ============================================================

METHODS = ["ordered", "original", "balanced"]


def _positive_finite(df, column):
    return np.isfinite(df[column]) & (df[column] > 0)


def plot_distribution_specific_results(summary, hypothesis):
    """
    For every distribution:
      A) n vs aggregated B_eq
      B) n vs formula runtime + equivalent-MC runtime
    """

    subset = summary.loc[
        summary["hypothesis"] == hypothesis
    ].copy()

    for distribution, dist_data in subset.groupby(
        "distribution", sort=False
    ):

        # -----------------------------------------------
        # A. B_eq
        # -----------------------------------------------
        fig, axes = plt.subplots(
            1, 3,
            figsize=(16, 4.8),
            sharex=True,
            sharey=False,
        )

        for ax, method in zip(axes, METHODS):
            d = dist_data.loc[
                dist_data["method"] == method
            ].sort_values("n")

            valid = _positive_finite(d, "aggregated_B_eq")

            ax.plot(
                d.loc[valid, "n"],
                d.loc[valid, "aggregated_B_eq"],
                marker="o",
                linewidth=2.2,
            )

            # Annotate unresolved scenario n values.
            unresolved = d["B_eq_status"] == "unresolved_reference"
            for n_bad in d.loc[unresolved, "n"]:
                ax.axvline(
                    n_bad,
                    linestyle=":",
                    linewidth=0.8,
                    alpha=0.25,
                )

            ax.set_xscale("log")
            if valid.any():
                ax.set_yscale("log")
            else:
                ax.text(
                    0.5, 0.5,
                    "No resolved $B_{eq}$ points",
                    transform=ax.transAxes,
                    ha="center", va="center",
                    fontsize=9, alpha=0.7,
                )
            ax.set_xlabel("Sample size per group, n")
            ax.set_title(method.capitalize())
            ax.grid(True, which="both", alpha=0.25)

        axes[0].set_ylabel("Aggregated equivalent MC permutations")

        fig.suptitle(
            f"{distribution} — {hypothesis}: equivalent MC budget",
            fontsize=14,
            y=1.03,
        )

        plt.tight_layout()
        save_or_show_figure(
            fig,
            f"beq_{_safe_filename(hypothesis)}_{_safe_filename(distribution)}.png",
        )

        # -----------------------------------------------
        # B. Equivalent time
        # -----------------------------------------------
        fig, axes = plt.subplots(
            1, 3,
            figsize=(16, 4.8),
            sharex=True,
            sharey=True,
        )

        for ax, method in zip(axes, METHODS):
            d = dist_data.loc[
                dist_data["method"] == method
            ].sort_values("n")

            valid_formula = _positive_finite(
                d, "median_formula_time_seconds"
            )
            valid_mc = _positive_finite(
                d, "equivalent_mc_time_seconds"
            )

            ax.plot(
                d.loc[valid_formula, "n"],
                d.loc[valid_formula, "median_formula_time_seconds"],
                marker="s",
                linewidth=2.5,
                label="Formula runtime",
            )

            ax.plot(
                d.loc[valid_mc, "n"],
                d.loc[valid_mc, "equivalent_mc_time_seconds"],
                marker="o",
                linestyle="--",
                linewidth=2.2,
                label="MC runtime at equal MSE",
            )

            ax.set_xscale("log")
            ax.set_yscale("log")
            ax.set_xlabel("Sample size per group, n")
            ax.set_title(method.capitalize())
            ax.grid(True, which="both", alpha=0.25)
            ax.legend(fontsize=8)

        axes[0].set_ylabel("Runtime (seconds)")

        fig.suptitle(
            f"{distribution} — {hypothesis}: formula vs equivalent-MC runtime",
            fontsize=14,
            y=1.03,
        )

        plt.tight_layout()
        save_or_show_figure(
            fig,
            f"runtime_{_safe_filename(hypothesis)}_{_safe_filename(distribution)}.png",
        )


def plot_all_distributions_Beq(summary, hypothesis):
    """
    One combined figure:
    all generating-distribution B_eq curves together.
    """
    subset = summary.loc[
        summary["hypothesis"] == hypothesis
    ].copy()

    fig, axes = plt.subplots(
        1, 3,
        figsize=(17, 5.2),
        sharex=True,
        sharey=False,
    )

    for ax, method in zip(axes, METHODS):
        method_data = subset.loc[
            subset["method"] == method
        ]

        any_valid = False

        for distribution, d in method_data.groupby(
            "distribution", sort=False
        ):
            d = d.sort_values("n")
            valid = _positive_finite(d, "aggregated_B_eq")
            any_valid = any_valid or bool(valid.any())

            ax.plot(
                d.loc[valid, "n"],
                d.loc[valid, "aggregated_B_eq"],
                marker="o",
                linewidth=2,
                label=distribution,
            )

        ax.set_xscale("log")
        if any_valid:
            ax.set_yscale("log")
        else:
            ax.text(
                0.5, 0.5,
                "No resolved $B_{eq}$ points",
                transform=ax.transAxes,
                ha="center", va="center",
                fontsize=9, alpha=0.7,
            )
        ax.set_xlabel("Sample size per group, n")
        ax.set_title(method.capitalize())
        ax.grid(True, which="both", alpha=0.25)

    axes[0].set_ylabel("Aggregated equivalent MC permutations")

    handles, labels = axes[-1].get_legend_handles_labels()

    fig.legend(
        handles,
        labels,
        loc="upper center",
        ncol=3,
        bbox_to_anchor=(0.5, 1.04),
    )

    fig.suptitle(
        f"Effect of generating distribution on B_eq — {hypothesis}",
        fontsize=14,
        y=1.12,
    )

    plt.tight_layout()
    save_or_show_figure(
        fig,
        f"beq_all_distributions_{_safe_filename(hypothesis)}.png",
    )


def plot_all_distributions_time(summary, hypothesis):
    """
    One combined figure:
      - one solid formula-runtime curve per method
      - one dashed equivalent-MC curve per distribution
    """
    subset = summary.loc[
        summary["hypothesis"] == hypothesis
    ].copy()

    fig, axes = plt.subplots(
        1, 3,
        figsize=(17, 5.2),
        sharex=True,
        sharey=True,
    )

    for ax, method in zip(axes, METHODS):
        method_data = subset.loc[
            subset["method"] == method
        ].copy()

        # Formula runtime pooled across generating distributions.
        formula_curve = (
            method_data.groupby("n", as_index=False)
            ["median_formula_time_seconds"]
            .median()
            .sort_values("n")
        )

        valid_formula = _positive_finite(
            formula_curve,
            "median_formula_time_seconds",
        )

        ax.plot(
            formula_curve.loc[valid_formula, "n"],
            formula_curve.loc[
                valid_formula,
                "median_formula_time_seconds",
            ],
            marker="s",
            linewidth=3.0,
            label=f"{method.capitalize()} formula",
        )

        for distribution, d in method_data.groupby(
            "distribution", sort=False
        ):
            d = d.sort_values("n")
            valid = _positive_finite(
                d, "equivalent_mc_time_seconds"
            )

            ax.plot(
                d.loc[valid, "n"],
                d.loc[valid, "equivalent_mc_time_seconds"],
                marker="o",
                linestyle="--",
                linewidth=2,
                label=distribution,
            )

        ax.set_xscale("log")
        ax.set_yscale("log")
        ax.set_xlabel("Sample size per group, n")
        ax.set_title(method.capitalize())
        ax.grid(True, which="both", alpha=0.25)

    axes[0].set_ylabel("Runtime (seconds)")

    handles, labels = axes[-1].get_legend_handles_labels()

    fig.legend(
        handles,
        labels,
        loc="upper center",
        ncol=3,
        bbox_to_anchor=(0.5, 1.05),
    )

    fig.suptitle(
        f"Equivalent MC runtime across generating distributions — {hypothesis}",
        fontsize=14,
        y=1.13,
    )

    plt.tight_layout()
    save_or_show_figure(
        fig,
        f"runtime_all_distributions_{_safe_filename(hypothesis)}.png",
    )


def plot_reference_resolution(summary, hypothesis):
    """
    Useful cluster diagnostic:
    number of unresolved reference datasets out of R.
    """
    subset = (
        summary.loc[
            summary["hypothesis"] == hypothesis
        ]
        .groupby(
            ["distribution", "n"],
            as_index=False,
        )
        ["unresolved_references"]
        .max()
    )

    pivot = subset.pivot(
        index="distribution",
        columns="n",
        values="unresolved_references",
    )

    fig, ax = plt.subplots(figsize=(9, 5.5))
    image = ax.imshow(
        pivot.to_numpy(),
        aspect="auto",
        origin="upper",
    )

    ax.set_xticks(np.arange(len(pivot.columns)))
    ax.set_xticklabels([str(int(n)) for n in pivot.columns])

    ax.set_yticks(np.arange(len(pivot.index)))
    ax.set_yticklabels(pivot.index)

    for i in range(pivot.shape[0]):
        for j in range(pivot.shape[1]):
            value = pivot.iloc[i, j]
            if np.isfinite(value):
                ax.text(
                    j, i,
                    f"{int(value)}/{R}",
                    ha="center",
                    va="center",
                )

    ax.set_xlabel("Sample size per group, n")
    ax.set_ylabel("Distribution")
    ax.set_title(
        f"Unresolved MC references (K=0) — {hypothesis}"
    )

    cbar = plt.colorbar(image, ax=ax)
    cbar.set_label("Number unresolved")

    plt.tight_layout()
    save_or_show_figure(
        fig,
        f"reference_resolution_{_safe_filename(hypothesis)}.png",
    )


In [ ]:

# ============================================================
# 11. RUN THE STUDY
# ============================================================
#
# If the run is interrupted, rerun this cell with resume=True.
# Completed scenarios are read from the checkpoint and skipped.
# ============================================================

STUDY_START = perf_counter()

raw_results = run_multidistribution_study(
    distributions=DISTRIBUTIONS,
    hypotheses=HYPOTHESES,
    n_values=N_VALUES,
    R=R,
    B_ref=B_REF,
    batch_size=BATCH_SIZE,
    master_seed=MASTER_SEED,
    tail=TAIL,
    min_extreme_count=MIN_EXTREME_COUNT,

    mean_x_h0=MEAN_X_H0,
    mean_y_h0=MEAN_Y_H0,

    mean_x_h1=MEAN_X_H1,
    mean_y_h1=MEAN_Y_H1,

    standard_deviation=STANDARD_DEVIATION,

    checkpoint_path=CHECKPOINT_RAW,
    resume=True,
)

STUDY_RUNTIME_SECONDS = perf_counter() - STUDY_START

summary_results = summarize_distribution_study(
    raw_results,
    strict_unresolved=True,
)

summary_results.to_csv(
    SUMMARY_FILE,
    index=False,
)

generation_manifest = build_generation_manifest(
    raw_results,
    distributions=DISTRIBUTIONS,
    n_values=N_VALUES,

    mean_x_h0=MEAN_X_H0,
    mean_y_h0=MEAN_Y_H0,

    mean_x_h1=MEAN_X_H1,
    mean_y_h1=MEAN_Y_H1,

    standard_deviation=STANDARD_DEVIATION,
)

generation_manifest.to_csv(
    GENERATION_FILE,
    index=False,
)

print("\nFinished.")
print(f"Raw rows:       {len(raw_results):,}")
print(f"Summary rows:   {len(summary_results):,}")
print(f"Generation rows:{len(generation_manifest):,}")

summary_view = summary_results[
    [
        "hypothesis",
        "distribution",
        "n",
        "method",
        "R",
        "well_resolved_references",
        "low_resolution_references",
        "unresolved_references",
        "B_eq_status",
        "aggregated_B_eq",
        "equivalent_mc_time_seconds",
    ]
]

if IS_PBS_JOB:
    print(summary_view.to_string(index=False))
else:
    display(summary_view)

print(f"Study runtime: {STUDY_RUNTIME_SECONDS/60:.2f} minutes")


In [ ]:

# ============================================================
# 12. PLOT GENERATED DATASETS
# ============================================================

for hypothesis in HYPOTHESES:
    plot_generated_distributions(
        generation_manifest,
        distributions=DISTRIBUTIONS,
        hypothesis=hypothesis,
        n_values=N_VALUES,

        mean_x_h0=MEAN_X_H0,
        mean_y_h0=MEAN_Y_H0,

        mean_x_h1=MEAN_X_H1,
        mean_y_h1=MEAN_Y_H1,

        standard_deviation=STANDARD_DEVIATION,
    )


In [ ]:

# ============================================================
# 13. PLOT STUDY RESULTS
# ============================================================

for hypothesis in HYPOTHESES:

    print("\n" + "=" * 80)
    print(f"RESULTS: {hypothesis}")
    print("=" * 80)

    # One distribution at a time.
    plot_distribution_specific_results(
        summary_results,
        hypothesis=hypothesis,
    )

    # All distributions together.
    plot_all_distributions_Beq(
        summary_results,
        hypothesis=hypothesis,
    )

    plot_all_distributions_time(
        summary_results,
        hypothesis=hypothesis,
    )

    # Resolution diagnostic.
    plot_reference_resolution(
        summary_results,
        hypothesis=hypothesis,
    )


In [ ]:

# ============================================================
# 14. SANITY CHECKS
# ============================================================

def run_sanity_checks():
    rng = np.random.default_rng(12345)

    # Moderate-probability agreement between ordinary and log-space algebra.
    p_ref = 0.2
    p_formula = 0.18

    direct = (
        p_ref * (1.0 - p_ref)
        / (p_formula - p_ref)**2
    )

    log_num = log_p_one_minus_p(np.log(p_ref))
    log_err = 2.0 * log_abs_exp_difference(
        np.log(p_formula),
        np.log(p_ref),
    )
    logspace = np.exp(log_num - log_err)

    assert np.allclose(direct, logspace, rtol=1e-12)

    # EC weights sum to one.
    for n in [10, 100, 500]:
        _, lw = ec_log_weights(n)
        assert np.allclose(np.exp(lw).sum(), 1.0)

    # Balanced split really has n observations in each artificial group.
    x = rng.normal(size=20)
    y = rng.normal(size=20)

    a, b = build_balanced_split(x, y)
    assert len(a) == len(x)
    assert len(b) == len(y)

    # Small end-to-end formula smoke check.
    for estimator in EC_ESTIMATORS.values():
        result = estimator(x, y)
        assert np.isfinite(result["log_p_value"])
        assert result["log_p_value"] <= 0.0

    print("All sanity checks passed.")


run_sanity_checks()


# ============================================================
# 15. SAVE RUN METADATA + CLEAN TEMPORARY SCRATCH
# ============================================================

TOTAL_RUNTIME_SECONDS = perf_counter() - NOTEBOOK_START

run_metadata = {
    "start_utc": START_UTC,
    "finish_utc": datetime.now(timezone.utc).isoformat(),
    "total_runtime_seconds": float(TOTAL_RUNTIME_SECONDS),
    "study_runtime_seconds": float(globals().get("STUDY_RUNTIME_SECONDS", np.nan)),
    "is_pbs_job": bool(IS_PBS_JOB),
    "pbs_job_id": str(PBS_JOB_ID),
    "pbs_queue": str(PBS_QUEUE),
    "hostname": str(HOSTNAME),
    "user": str(USER_NAME),
    "project_dir": str(PROJECT_DIR),
    "output_dir": str(OUTPUT_DIR),
    "figure_dir": str(FIGURE_DIR),
    "temp_dir": str(TEMP_DIR),
    "run_name": RUN_NAME,
    "run_hash": RUN_HASH,
    "configuration": RUN_CONFIG,
}

with open(METADATA_FILE, "w", encoding="utf-8") as f:
    json.dump(run_metadata, f, indent=2, sort_keys=True)

print("\nRun metadata")
print("------------")
print(f"Total notebook runtime = {TOTAL_RUNTIME_SECONDS/60:.2f} minutes")
print(f"Metadata file          = {METADATA_FILE}")
print(f"Persistent results     = {OUTPUT_DIR}")

# The notebook currently has no heavy temporary I/O, but a unique
# scratch directory is created for future extensions. Remove it only
# when empty / after all persistent outputs have been written.
try:
    if TEMP_DIR.exists():
        shutil.rmtree(TEMP_DIR)
        print(f"Removed temporary scratch directory: {TEMP_DIR}")
except Exception as exc:
    warnings.warn(f"Could not remove temporary scratch directory: {exc}")



## OpenPBS notes for this notebook

The notebook is now dual-mode:

- **Local:** output goes to `./distribution_study_outputs/<run_name>/` and figures are both saved and displayed.
- **PBS compute job:** output goes to `/work/<user>/thesis_results/multidistribution/<run_name>/`; figures are saved without GUI display.

For the first cluster benchmark, keep the full-run settings identical to the PC pilot (`R=20`, `B_REF=10_000`, same `N_VALUES`, H0/H1, same six distributions). The metadata JSON records both the numerical study runtime and the total notebook runtime so that the PC-vs-cluster speed comparison is reproducible.

The configuration hash in the output folder name means that changing important hyperparameters automatically creates a fresh run directory rather than accidentally resuming an incompatible checkpoint.
